# Imports

In [2]:
import numpy as np
import pandas as pd
import category_encoders as ce

from lightgbm import LGBMClassifier

from scipy.optimize import minimize

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from sklearn.model_selection import cross_val_predict, StratifiedKFold, cross_val_score

from sklearn.utils.validation import check_X_y, check_array, check_is_fitted

## Utils

In [3]:
class MulticlassThresholdOptimizer(BaseEstimator, ClassifierMixin):
    def __init__(self, n_splits=5, method='Nelder-Mead', maxiter=500, random_state=42):
        self.n_splits = n_splits
        self.method = method
        self.maxiter = maxiter
        self.random_state = random_state
        
    def _objective(self, weights, X, y):
        weighted_probs = X * weights
        predictions = np.argmax(weighted_probs, axis=1)
        score = balanced_accuracy_score(y, predictions)
        return -score

    def fit(self, X, y):
        X, y = check_X_y(X, y, ensure_2d=True, force_all_finite=True)
        
        self.n_classes_ = X.shape[1]
        
        if self.n_splits is None or self.n_splits <= 1:
            initial_weights = np.ones(self.n_classes_)
            res = minimize(self._objective, initial_weights, args=(X, y), method=self.method, options={'maxiter': self.maxiter})
            self.weights_ = res.x
            return self
        
        kf = StratifiedKFold(n_splits=self.n_splits, shuffle=True, random_state=self.random_state)
        fold_weights = []
        
        for train_idx, val_idx in kf.split(X, y):
            X_train, y_train = X[train_idx], y[train_idx]
            
            initial_weights = np.ones(self.n_classes_)
            res = minimize(self._objective, initial_weights, args=(X_train, y_train), method=self.method, options={'maxiter': self.maxiter})
            
            fold_weights.append(res.x)
        
        self.weights_ = np.mean(fold_weights, axis=0)
        return self

    def predict_proba(self, X):
        check_is_fitted(self, attributes=['weights_'])
        X = check_array(X, ensure_2d=True)
        
        weighted_probs = X * self.weights_

        sum_probs = np.sum(weighted_probs, axis=1, keepdims=True)
        sum_probs = np.where(sum_probs == 0, 1, sum_probs) 
        
        return weighted_probs / sum_probs

    def predict(self, X):
        check_is_fitted(self, attributes=['weights_'])
        X = check_array(X, ensure_2d=True)
        
        weighted_probs = X * self.weights_
        return np.argmax(weighted_probs, axis=1)

# Loading Dataset

In [4]:
X_train_raw = pd.read_parquet('../data/X_train_raw.parquet')
X_train_fe = pd.read_parquet('../data/X_train_fe.parquet')
y_train = pd.read_parquet('../data/y_train.parquet')

X_test_raw = pd.read_parquet('../data/X_test_raw.parquet')
X_test_fe = pd.read_parquet('../data/X_test_fe.parquet')

In [5]:
X_train_raw.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence


In [6]:
X_train_fe.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,...,g_sub_r,r_sub_i,i_sub_z,u_sub_r,u_sub_z,redshift_mult_r,redshift_mult_i,x_espatial_coordinates,y_espatial_coordinates,z_espatial_coordinates
id,,,,,,,,,,,,,,,,,,,,,
0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,M,Red_Sequence,...,1.537632,1.100813,0.636056,5.114196,6.851065,8.326031,7.875818,0.313090,0.024912,-0.949397
1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,M,Red_Sequence,...,1.499854,0.361141,0.439634,3.191300,3.992076,2.778353,2.721302,-0.408896,0.435262,0.802092
2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,O/B,Blue_Cloud,...,-0.092712,0.589211,0.025263,-0.136637,0.477837,59.784407,58.120611,0.529713,0.466346,-0.708467
3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,M,Red_Sequence,...,2.032982,0.652096,0.450706,4.287302,5.390104,10.195392,9.845805,-0.116193,0.045921,-0.992165
4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,M,Red_Sequence,...,1.237231,0.335002,0.283262,3.468709,4.086973,10.134002,9.947821,-0.787468,-0.394539,0.473532


In [7]:
X_test_raw.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population
id,,,,,,,,,,
577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence
577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence
577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud
577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence
577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence


In [8]:
X_test_fe.head()

,alpha,delta,u,g,r,i,z,redshift,spectral_type,galaxy_population,...,g_sub_r,r_sub_i,i_sub_z,u_sub_r,u_sub_z,redshift_mult_r,redshift_mult_i,x_espatial_coordinates,y_espatial_coordinates,z_espatial_coordinates
id,,,,,,,,,,,,,,,,,,,,,
577347,120.719779,23.924249,23.668066,21.951680,21.086183,20.180032,19.202124,0.429042,G/K,Red_Sequence,...,0.865497,0.906151,0.977908,2.581883,4.465942,9.046867,8.658089,0.081333,0.344971,-0.935083
577348,219.414419,42.171651,24.902933,22.338822,20.732163,19.860330,19.687691,0.867305,M,Red_Sequence,...,1.606658,0.871833,0.172640,4.170770,5.215243,17.981106,17.224961,-0.208809,0.113279,-0.971374
577349,173.568731,-1.756400,19.427591,18.474633,17.551314,16.570674,16.176765,0.224234,G/K,Blue_Cloud,...,0.923319,0.980640,0.393909,1.876277,3.250826,3.935608,3.715715,0.131045,0.129932,-0.982825
577350,184.903993,-1.411074,23.121029,21.526855,20.670159,20.417633,20.699095,0.066507,G/K,Red_Sequence,...,0.856696,0.252526,-0.281462,2.450870,2.421934,1.374717,1.357922,-0.143212,0.069175,-0.987272
577351,222.487816,15.381403,25.094282,22.643981,21.123173,19.439500,19.094158,0.977218,M,Red_Sequence,...,1.520808,1.683673,0.345342,3.971109,6.000123,20.641941,18.996626,0.799820,-0.507330,0.320787


In [7]:
y_train.head()

,class,class_encoded
id,,
0,GALAXY,0
1,GALAXY,0
2,QSO,1
3,GALAXY,0
4,GALAXY,0


# Machine Learning

## Base Model

In [ ]:
base_model = make_pipeline(
    ce.TargetEncoder(cols=['spectral_type', 'galaxy_population']),
    LGBMClassifier(
        objective='multiclass',
        metric='multi_logloss',
        num_class=3,
        boosting_type='gbdt',
        verbosity=-1,
        n_estimators=2000,
        random_state=42,
        n_jobs=1,
    )
)

cross_val_score(base_model, X_train_raw, y_train.class_encoded, cv=5, scoring='roc_auc_ovr', n_jobs=-1)

array([0.98558227, 0.98650875, 0.98230038, 0.984977  , 0.99078166])

In [13]:
np.mean([0.98558227, 0.98650875, 0.98230038, 0.984977, 0.99078166])

np.float64(0.986030012)

In [16]:
np.std([0.98558227, 0.98650875, 0.98230038, 0.984977, 0.99078166])

np.float64(0.0027577821094016968)

## Feature Engineering Model

In [12]:
cross_val_score(base_model, X_train_fe, y_train.class_encoded, cv=5, scoring='roc_auc_ovr', n_jobs=-1)

array([0.98367304, 0.97923879, 0.98472324, 0.98358016, 0.9846533 ])

In [ ]:
np.mean([0.98367304, 0.97923879, 0.98472324, 0.98358016, 0.9846533])

np.float64(0.983173706)

In [15]:
np.std([0.98367304, 0.97923879, 0.98472324, 0.98358016, 0.9846533])

np.float64(0.0020242705968679035)